# Face Recognition Prototype
- populate face DB
  - given a video file, 
    - detects unique faces
    - store into DB
- identify face from video


## Install pre-requisites

In [1]:
# on mac, new to also `brew install cmake`
!pip install face_recognition opencv-python numpy

## Import Python Modules

In [2]:
import cv2
import face_recognition
import sqlite3
import numpy as np
import os
import shutil
from datetime import datetime

## Configuration

In [3]:
DB_PATH = "face_data.db"
IMAGE_STORAGE = "detected_faces"
os.makedirs(IMAGE_STORAGE, exist_ok=True)

## DB Utility Functions

In [4]:
def get_db_connection():
    conn = sqlite3.connect(DB_PATH)
    return conn

In [5]:
def init_db():
    conn = get_db_connection()
    cursor = conn.cursor()
    # Stores unique face identities and their embeddings
    cursor.execute('''CREATE TABLE IF NOT EXISTS unique_faces 
                      (id INTEGER PRIMARY KEY, embedding BLOB)''')
    # Stores paths to sample images (up to 5 per face)
    cursor.execute('''CREATE TABLE IF NOT EXISTS face_samples 
                      (face_id INTEGER, image_path TEXT)''')
    # Stores video filenames associated with a face
    cursor.execute('''CREATE TABLE IF NOT EXISTS video_sources 
                      (face_id INTEGER, video_name TEXT, UNIQUE(face_id, video_name))''')
    conn.commit()
    conn.close()

In [6]:
def backup_database():
    backup_dir = "backups"
    os.makedirs(backup_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(backup_dir, f"face_data_{timestamp}.db")
    
    # Copy DB file
    shutil.copy2(DB_PATH, backup_path)
    
    # Keep only top 3
    all_backups = sorted([os.path.join(backup_dir, f) for f in os.listdir(backup_dir)], 
                         key=os.path.getctime, reverse=True)
    
    if len(all_backups) > 3:
        for old_backup in all_backups[3:]:
            os.remove(old_backup)
            print(f"Deleted old backup: {old_backup}")

## Face Analyzes and Storage Utility Functions

In [7]:
def process_and_store_video(video_path):
    conn = get_db_connection()
    video_name = os.path.basename(video_path)
    cap = cv2.VideoCapture(video_path)
    
    # Load existing faces from DB
    cursor = conn.cursor()
    cursor.execute("SELECT id, embedding FROM unique_faces")
    known_data = cursor.fetchall()
    known_ids = [row[0] for row in known_data]
    known_encodings = [np.frombuffer(row[1]) for row in known_data]

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # Process every 10th frame to save CPU
        if frame_count % 10 == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face_locations = face_recognition.face_locations(rgb_frame)
            face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)

            for encoding, location in zip(face_encodings, face_locations):
                match_id = None
                if known_encodings:
                    matches = face_recognition.compare_faces(known_encodings, encoding, tolerance=0.6)
                    if True in matches:
                        match_id = known_ids[matches.index(True)]

                if match_id is None:
                    # New Face Detected
                    cursor.execute("INSERT INTO unique_faces (embedding) VALUES (?)", (encoding.tobytes(),))
                    match_id = cursor.lastrowid
                    known_ids.append(match_id)
                    known_encodings.append(encoding)
                
                # Update Video Source
                cursor.execute("INSERT OR IGNORE INTO video_sources (face_id, video_name) VALUES (?, ?)", 
                               (match_id, video_name))

                # Save Sample Image (if less than 5)
                cursor.execute("SELECT COUNT(*) FROM face_samples WHERE face_id = ?", (match_id,))
                if cursor.fetchone()[0] < 5:
                    top, right, bottom, left = location
                    face_img = frame[top:bottom, left:right]
                    img_path = f"{IMAGE_STORAGE}/face_{match_id}_{datetime.now().strftime('%f')}.jpg"
                    cv2.imwrite(img_path, face_img)
                    cursor.execute("INSERT INTO face_samples (face_id, image_path) VALUES (?, ?)", 
                                   (match_id, img_path))
        
        frame_count += 1
    
    conn.commit()
    conn.close()
    cap.release()

In [8]:
def search_faces_in_video(video_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT id, embedding FROM unique_faces")
    db_faces = cursor.fetchall()
    
    known_ids = [row[0] for row in db_faces]
    known_encodings = [np.frombuffer(row[1]) for row in db_faces]
    
    cap = cv2.VideoCapture(video_path)
    found_matches = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        face_encodings = face_recognition.face_encodings(rgb_frame)

        for encoding in face_encodings:
            if known_encodings:
                matches = face_recognition.compare_faces(known_encodings, encoding)
                for i, match in enumerate(matches):
                    if match:
                        found_matches.add(known_ids[i])

    print(f"Detected {len(found_matches)} recognized faces from DB:")
    for f_id in found_matches:
        cursor.execute("SELECT video_name FROM video_sources WHERE face_id = ?", (f_id,))
        videos = [v[0] for v in cursor.fetchall()]
        print(f" - Face ID {f_id}: Previously seen in {videos}")
    
    conn.close()
    cap.release()

## Execute

In [9]:
init_db()

In [10]:
process_and_store_video("personA.mp4")

In [11]:
process_and_store_video("personB.mp4")

In [12]:
search_faces_in_video("test_video_A.mp4")

Detected 2 recognized faces from DB:
 - Face ID 1: Previously seen in ['personA.mp4']
 - Face ID 2: Previously seen in ['personB.mp4']


In [13]:
search_faces_in_video("test_video_B.mp4")

Detected 2 recognized faces from DB:
 - Face ID 1: Previously seen in ['personA.mp4']
 - Face ID 2: Previously seen in ['personB.mp4']


In [14]:
search_faces_in_video("test_video_empty.mp4")

Detected 0 recognized faces from DB:


## Credit
- Sample video (to cross-check transcription)
    - [Learn Japanese with Short Dramas - WAKU WAKU Japanese](https://www.youtube.com/watch?v=i2UEIUI3XRI)